# Classify Reddit posts with multiMentalRoBERTA-6-class

Runs [SajjadIslam/multiMentalRoBERTA-6-class](https://huggingface.co/SajjadIslam/multiMentalRoBERTA-6-class) on every post in `data/redditPosts.json` and shows the top 2 labels with confidence, next to the labels currently stored in the file.

Uses the same pipeline settings as the backend's `/posts/detect-labels` endpoint. The last cell writes the model's labels back into `redditPosts.json`; every cell before it only reads the file.

Select a kernel that has `transformers`, `torch` and `pandas` installed (e.g. the `py312_gpu_2` conda environment).

In [1]:
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from transformers import pipeline

MODEL = "SajjadIslam/multiMentalRoBERTA-6-class"
TOP_K = 2
MAX_TOKENS = 512

# Works whether the notebook is started from tests/ or from PSS-FastAPI-Backend/
BACKEND_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "redditPosts.json").exists())
POSTS_FILE = BACKEND_DIR / "data" / "redditPosts.json"
load_dotenv(BACKEND_DIR / ".env")

pd.set_option("display.max_colwidth", None)
print(POSTS_FILE)

d:\Sajjad-Workspace\PSS-LLM-Evaluation\PSS-FastAPI-Backend\data\redditPosts.json


In [2]:
posts = json.loads(POSTS_FILE.read_text(encoding="utf-8"))
print(f"Loaded {len(posts)} posts")

Loaded 5 posts


In [3]:
# First run downloads the model (~500 MB); later runs use the local cache.
classifier = pipeline(
    "text-classification",
    model=MODEL,
    top_k=None,
    truncation=True,
    token=os.getenv("HUGGINGFACE_TOKEN") or None,
)
print("Model labels:", list(classifier.model.config.id2label.values()))

Device set to use cuda:0


Model labels: ['anxiety', 'depression', 'ptsd', 'suicide', 'stress', 'none']


In [4]:
def format_labels(labels):
    return ", ".join(f"{l['name']} {l['percentage']}%" for l in labels) or "-"


results = classifier([post["content"] for post in posts])

rows = []
predictions = {}
for post, scores in zip(posts, results):
    top = sorted(scores, key=lambda s: s["score"], reverse=True)[:TOP_K]
    predictions[post["id"]] = top
    token_count = len(classifier.tokenizer(post["content"])["input_ids"])
    rows.append({
        "id": post["id"],
        "post (start)": post["content"][:80] + "...",
        "model label 1": f"{top[0]['label']} ({top[0]['score']:.2%})",
        "model label 2": f"{top[1]['label']} ({top[1]['score']:.2%})",
        "labels in JSON": format_labels(post.get("labels", [])),
        "tokens": token_count,
        "truncated": token_count > MAX_TOKENS,
    })

df = pd.DataFrame(rows).set_index("id")
df

Token indices sequence length is longer than the specified maximum sequence length for this model (672 > 512). Running this sequence through the model will result in indexing errors


,post (start),model label 1,model label 2,labels in JSON,tokens,truncated
id,,,,,,
1,"Back in April my husband had a serious mental break - he left me, his wife, and ...",depression (99.48%),suicide (0.36%),"Stress 46%, Depression 34%",270,False
2,"Hey all, former Army here! 7 years, got out in 2020 ( Covid Sucked!) and have be...",depression (94.53%),suicide (4.55%),"Stress 46%, Depression 34%",138,False
3,I spent 13 years active duty. Separated this past year due to medical reasons. S...,suicide (92.17%),depression (7.43%),"PTSD 34%, Anxiety 26%",672,True
4,I'm well aware ESA's and service animals are very different and follow very diff...,none (96.44%),depression (2.21%),"PTSD 34%, Anxiety 26%",390,False
5,"39M, Marine Vet.\n\nBeen feeling low, lost, and broken.\n\nI served from 2005-2016.\n...",depression (99.32%),suicide (0.61%),"Suicidal 70%, Stress 20%",158,False


## Labels in the app's format

The model returns lowercase names (e.g. `suicide`), while the app's selectable labels use `Suicidal`, `PTSD`, etc. This cell converts to the app's names and previews each post's `category` and `labels` values.

In [5]:
APP_LABEL_NAMES = {
    "anxiety": "Anxiety",
    "depression": "Depression",
    "ptsd": "PTSD",
    "suicide": "Suicidal",
    "stress": "Stress",
    "none": "None",
}

PERCENT_DECIMALS = 2

app_labels = {
    post_id: [
        {
            "name": APP_LABEL_NAMES.get(s["label"], s["label"]),
            "percentage": round(s["score"] * 100, PERCENT_DECIMALS),
        }
        for s in top
    ]
    for post_id, top in predictions.items()
}

for post_id, labels in app_labels.items():
    print(f"Post {post_id}")
    print(f'  "category": "{format_labels(labels)}",')
    print(f'  "labels": {json.dumps(labels)}')
    print()

Post 1
  "category": "Depression 99%, Suicidal 0%",
  "labels": [{"name": "Depression", "percentage": 99}, {"name": "Suicidal", "percentage": 0}]

Post 2
  "category": "Depression 95%, Suicidal 5%",
  "labels": [{"name": "Depression", "percentage": 95}, {"name": "Suicidal", "percentage": 5}]

Post 3
  "category": "Suicidal 92%, Depression 7%",
  "labels": [{"name": "Suicidal", "percentage": 92}, {"name": "Depression", "percentage": 7}]

Post 4
  "category": "None 96%, Depression 2%",
  "labels": [{"name": "None", "percentage": 96}, {"name": "Depression", "percentage": 2}]

Post 5
  "category": "Depression 99%, Suicidal 1%",
  "labels": [{"name": "Depression", "percentage": 99}, {"name": "Suicidal", "percentage": 1}]



## Write the labels into `redditPosts.json`

Updates each post's `category` and `labels` with the model's top 2 labels from above. All other fields (`id`, `content`, `default-response`) are kept as they are. The Home page shows these labels as the "AI detected mental state labels".

The file is tracked in git, so `git diff` shows the change and `git checkout -- data/redditPosts.json` undoes it.

In [ ]:
# Re-read the file so edits made since the posts were loaded are not overwritten.
posts_on_disk = json.loads(POSTS_FILE.read_text(encoding="utf-8"))

updated = 0
for post in posts_on_disk:
    labels = app_labels.get(post["id"])
    if labels is None:
        print(f"Post {post['id']}: not classified in this run, left unchanged")
        continue
    post["category"] = format_labels(labels)
    post["labels"] = labels
    updated += 1

POSTS_FILE.write_text(json.dumps(posts_on_disk, indent="\t", ensure_ascii=False) + "\n", encoding="utf-8")
print(f"Updated {updated} of {len(posts_on_disk)} posts in {POSTS_FILE}")

pd.DataFrame(
    [{"id": p["id"], "category": p.get("category")} for p in posts_on_disk]
).set_index("id")